<a href="https://colab.research.google.com/github/musab855/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

Full-depth companion to ML-07 (`w04_baseline_score.ipynb`, which already carries the two
verdicts the rule needed): three signals, one bucket table each with **n printed**, one
one-word verdict apiece (CONFIRMED / OPPOSITE / MIXED / FALSE), then the flag-linked test
of FlyRank's `HIGH_VOLUME_POOR_POSITION` assumption. A clearly-explained negative is a win —
it saves the rule before it ships.

> Skills: `auditing-signals` + `flyrank/flyrank-data` (see `skills/README.md`).

## 1. Distributions

*Look before deciding. Search/traffic metrics are heavy-tailed — a few giants, a long tail
of tiny pages — and that one fact changes every test below: bucket tables with visible n,
never raw Pearson correlation on the raw values.*

In [1]:
# ---- Setup (repo-root aware; token never printed, stored, or written) --------
import os, sys, getpass, subprocess, importlib.util, shutil, json
from pathlib import Path

def find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "work").is_dir() and (p / "skills").is_dir():
            return p
    return None

root = find_repo_root(Path.cwd())
if root is None:
    if Path("/content").exists():
        dest = Path("/content/flyrank-ml-internship")   # Colab
    else:
        dest = Path.cwd() / "flyrank-ml-internship"
    if (dest / "work").is_dir() and (dest / "skills").is_dir():
        print(f"repo folder already present - reusing it: {dest}")
        root = dest
    else:
        if dest.exists():
            shutil.rmtree(dest)
        proc = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/musab855/flyrank-ml-internship.git", str(dest)],
            capture_output=True, text=True)
        if proc.returncode != 0:
            print(proc.stdout); print(proc.stderr)
            raise RuntimeError(f"git clone failed (exit {proc.returncode})")
        root = dest
os.chdir(root)

missing = [p for p in ("duckdb", "huggingface_hub") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    importlib.invalidate_caches()

def load_token():
    try:
        from google.colab import userdata
        for key in ("HF_TOKEN2", "HF_TOKEN"):
            try:
                token = userdata.get(key)
                if token:
                    return token
            except Exception:
                pass
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN"):
                return line.split("=", 1)[1].strip().strip(chr(34) + chr(39))
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import numpy as np
import pandas as pd
from IPython.display import display

print("working directory:", Path.cwd())

# ---- Feature frame: cache-first, same March 2026 window as Weeks 4-7 --------
CACHE = Path("work/outputs/feature_frame_march_audit.csv")

if CACHE.exists():
    df = pd.read_csv(CACHE)
    print(f"Cache hit: {len(df):,} rows from {CACHE}")
else:
    HF_TOKEN = load_token()
    import duckdb
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    df = con.sql(f"""
    WITH daily_averages AS (
      SELECT content_hash_id, client_hash_id,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) AS gsc_impressions_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) AS gsc_clicks_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) AS position_1to15,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) AS gsc_impressions_16to31
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
      GROUP BY content_hash_id, client_hash_id
    )
    SELECT content_hash_id AS content_id, client_hash_id,
      gsc_impressions_1to15 AS gsc_impressions_avg,
      gsc_clicks_1to15 AS gsc_clicks_avg,
      position_1to15 AS position_avg,
      gsc_impressions_16to31,
      (CASE WHEN gsc_impressions_16to31 IS NULL OR gsc_impressions_1to15 IS NULL THEN NULL
            WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1 ELSE 0 END) AS is_declining_label
    FROM daily_averages
    """).df()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    print(f"Scanned the warehouse (March 2026): {len(df):,} rows -> cached at {CACHE}")

n_before = len(df)
pos_nan, pos_zero = df["position_avg"].isna(), df["position_avg"] == 0
lab_nan = df["is_declining_label"].isna()
n_pos_nan, n_pos_zero, n_lab_nan_extra = int(pos_nan.sum()), int(pos_zero.sum()), int((lab_nan & ~pos_nan & ~pos_zero).sum())
print(f"Rows before filter: {n_before:,}")
print(f"Excluding position NaN (no GSC data days 1-15): {n_pos_nan:,}")
print(f"Excluding position == 0 (the 'no data' placeholder): {n_pos_zero:,}")
print(f"Excluding undefined labels (one half unmeasurable): {n_lab_nan_extra:,}")

df = df[~pos_nan & ~pos_zero & ~lab_nan].copy()
df["is_declining_label"] = df["is_declining_label"].astype(int)
df = df.sort_values("content_id", kind="stable").reset_index(drop=True)
assert len(df) + n_pos_nan + n_pos_zero + n_lab_nan_extra == n_before

BASE_FEATURES = ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]

# Frame receipts (work/outputs/capstone_metrics.json): assert, don't assume.
cm = json.loads(Path("work/outputs/capstone_metrics.json").read_text(encoding="utf-8"))
assert len(df) == cm["frame"]["pages"] == 140_599, f"frame changed: {len(df):,}"
assert df["client_hash_id"].nunique() == cm["frame"]["clients"] == 43
base_rate = float(df["is_declining_label"].mean())
assert abs(base_rate - cm["frame"]["base_rate"]) < 0.0005, f"base rate {base_rate:.4f}"
flagged = (df["gsc_impressions_avg"] >= 500) & (df["position_avg"] >= 11)
assert int(flagged.sum()) == cm["frame"]["flagged_by_rule"] == 1_074

print(f"\nModeling frame: {len(df):,} pages, {df['client_hash_id'].nunique()} clients, "
      f"base rate {base_rate:.4f}, flagged {int(flagged.sum()):,}")
print("features:", BASE_FEATURES)

# The label's own window column exists in the cache only as leakage-bait for
# ML-05/ML-09 tests. This notebook never uses it - drop it now.
df = df.drop(columns=["gsc_impressions_16to31"])

# ---- Distributions of the fields we are about to test ------------------------
rows = []
for col in ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]:
    s = df[col]
    rows.append({"feature": col, "n": len(s), "median": round(float(s.median()), 2),
                 "mean": round(float(s.mean()), 2), "p99": round(float(s.quantile(0.99)), 2),
                 "max": round(float(s.max()), 2),
                 "mean/median": (round(float(s.mean()) / float(s.median()), 1) if float(s.median()) > 0 else None)})
dist = pd.DataFrame(rows).set_index("feature")
display(dist)

imp_ratio = float(df["gsc_impressions_avg"].mean() / df["gsc_impressions_avg"].median())
assert imp_ratio > 3, f"impressions no longer look heavy-tailed (ratio {imp_ratio:.1f}) - re-read the data"
print(f"heavy tails confirmed: impressions mean/median = {imp_ratio:.0f}x "
      f"(median {df['gsc_impressions_avg'].median():.1f}/day, max {df['gsc_impressions_avg'].max():,.0f}/day)")
print("so: bucket tables with n (below), rates always with denominators, no raw correlation.")

working directory: /content/flyrank-ml-internship


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanned the warehouse (March 2026): 176,738 rows -> cached at work/outputs/feature_frame_march_audit.csv
Rows before filter: 176,738
Excluding position NaN (no GSC data days 1-15): 24,757
Excluding position == 0 (the 'no data' placeholder): 1,306
Excluding undefined labels (one half unmeasurable): 10,076

Modeling frame: 140,599 pages, 43 clients, base rate 0.4649, flagged 1,074
features: ['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg']


,n,median,mean,p99,max,mean/median
feature,,,,,,
gsc_impressions_avg,140599,10.53,63.27,795.27,12428.85,6.0
gsc_clicks_avg,140599,0.00,0.20,2.87,184.23,NaN
position_avg,140599,8.50,15.87,79.56,310.00,1.9


heavy tails confirmed: impressions mean/median = 6x (median 10.5/day, max 12,429/day)
so: bucket tables with n (below), rates always with denominators, no raw correlation.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three claims, tested identically — claim, bucket table with **n**, one-word verdict, one
sentence of meaning. Verdict vocabulary: **CONFIRMED / OPPOSITE / MIXED / FALSE**. Floor:
no verdict from a bucket under 500 pages (skill: `auditing-signals`).

1. *The more impressions a page gets, the more likely it is in decline.* (volume is the
   signal behind FlyRank's quick-win flag)
2. *Pages at position ≥ 11 are more likely to be in decline.* (the rule's position cutoff)
3. *Older content declines more.* (the refresh story everyone tells)

In [2]:
# ============================ Test #1: impressions volume =====================
imp_bins, imp_labels = [0, 100, 500, 2000, np.inf], ["<100", "100-500", "500-2000", "2000+"]
imp_table = (
    df.assign(bucket=pd.cut(df["gsc_impressions_avg"], bins=imp_bins, labels=imp_labels, right=False))
      .groupby("bucket", observed=True)["is_declining_label"]
      .agg(n="count", pct_declining="mean")
)
imp_table["pct_declining"] = (imp_table["pct_declining"] * 100).round(1)
print("Test #1 - claim: higher impressions -> more decline")
display(imp_table)
PUB_IMP = [45.1, 54.4, 56.3, 61.8]
under_floor = [b for b in imp_table.index if imp_table.loc[b, "n"] < 500]
assert under_floor == ["2000+"], f"unexpected under-floor impressions buckets: {under_floor}"
for got, want in zip(imp_table["pct_declining"], PUB_IMP):
    assert abs(got - want) <= 0.05, f"impressions bucket drifted: {got} vs published {want}"
assert imp_table.drop(index="2000+")["pct_declining"].is_monotonic_increasing, "impressions buckets no longer rise"
verdict_1 = "CONFIRMED"
print(f"VERDICT #1: {verdict_1} - rises across every readable bucket "
      f"({' -> '.join(f'{v}%' for v in PUB_IMP)}); 2000+ (n=186) is under "
      "the 500-page floor - printed, not read.")

# ============================ Test #2: average position ========================
pos_bins, pos_labels = [0, 4, 11, 31, np.inf], ["1-3", "4-10", "11-30", "31+"]
pos_table = (
    df.assign(bucket=pd.cut(df["position_avg"], bins=pos_bins, labels=pos_labels, right=False))
      .groupby("bucket", observed=True)["is_declining_label"]
      .agg(n="count", pct_declining="mean")
)
pos_table["pct_declining"] = (pos_table["pct_declining"] * 100).round(1)
print("\nTest #2 - claim: position >= 11 -> more decline")
display(pos_table)
PUB_POS = [47.2, 48.0, 45.9, 42.5]
assert (pos_table["n"] >= 500).all(), "a position bucket is under the 500-page floor"
for got, want in zip(pos_table["pct_declining"], PUB_POS):
    assert abs(got - want) <= 0.05, f"position bucket drifted: {got} vs published {want}"
assert not pos_table["pct_declining"].is_monotonic_increasing, "position now looks monotonic - re-read"
verdict_2 = "MIXED"
print(f"VERDICT #2: {verdict_2} - non-monotonic "
      f"({' -> '.join(f'{v}%' for v in PUB_POS)}): deep pages decline LESS than the middle.")

# ============ Test #3: content age (starter CSV: different dataset+label) ======
STARTER = Path("data/raw/content_refresh_anonymized.csv")
s = pd.read_csv(STARTER)
s["down"] = (s["trend_direction"] == "down").astype(int)
starter_base = float(s["down"].mean())
s["age_bucket"] = pd.cut(s["content_age_days"], [0, 180, 365, 730],
                         labels=["<180 days", "180-365 days", "365+ days"])
s["fresh_bucket"] = pd.cut(s["days_since_last_update"], [-1, 30, 90, 180, 365, 10**6],
                           labels=["0-30", "31-90", "91-180", "181-365", "365+"])
print(f"\nTest #3 - claim: older content declines more (starter CSV, base rate {starter_base:.3f})")
age_tbl = s.groupby("age_bucket", observed=True)["down"].agg(decline_rate="mean", pages="count")
age_tbl["decline_rate"] = (age_tbl["decline_rate"] * 100).round(1)
display(age_tbl)

pb = json.loads(Path("work/outputs/playbook_metrics.json").read_text(encoding="utf-8"))
for key, want in pb["decay_refresh_starter_csv"]["by_age_bucket"].items():
    got_rate = float(age_tbl.loc[key, "decline_rate"]) / 100
    assert abs(got_rate - want["decline_rate"]) < 0.005, f"age bucket {key} drifted"
    assert int(age_tbl.loc[key, "pages"]) == want["pages"], f"age bucket n {key} drifted"
assert age_tbl["decline_rate"].is_monotonic_decreasing, "age no longer falls against the claim"
verdict_3 = "OPPOSITE"
print(f"VERDICT #3: {verdict_3} - decline FALLS with age "
      f"({' -> '.join(f'{v}%' for v in age_tbl['decline_rate'])}) vs base {starter_base:.1%}; "
      "the story 'older content decays more' runs against this slice.")

fresh_tbl = s.groupby("fresh_bucket", observed=True)["down"].agg(decline_rate="mean", pages="count")
fresh_tbl["readable"] = fresh_tbl["pages"] >= 500
print("\nFreshness buckets (sample-size floor 500 - under it, no verdict):")
display(fresh_tbl.round(3))
n_unreadable = int((~fresh_tbl["readable"]).sum())
assert n_unreadable == 3, f"expected 3 unreadable freshness buckets, got {n_unreadable}"
print(f"{n_unreadable} of 5 freshness buckets are under the floor: 'insufficient data', "
      "which is itself a finding - freshness does not read on this slice.")

Test #1 - claim: higher impressions -> more decline


,n,pct_declining
bucket,,
<100,119836,45.1
100-500,17658,54.4
500-2000,2919,56.3
2000+,186,61.8


VERDICT #1: CONFIRMED - rises across every readable bucket (45.1% -> 54.4% -> 56.3% -> 61.8%); 2000+ (n=186) is under the 500-page floor - printed, not read.

Test #2 - claim: position >= 11 -> more decline


,n,pct_declining
bucket,,
1-3,24768,47.2
4-10,57912,48.0
11-30,36654,45.9
31+,21265,42.5


VERDICT #2: MIXED - non-monotonic (47.2% -> 48.0% -> 45.9% -> 42.5%): deep pages decline LESS than the middle.

Test #3 - claim: older content declines more (starter CSV, base rate 0.542)


,decline_rate,pages
age_bucket,,
<180 days,62.7,12272
180-365 days,51.5,11368
365+ days,42.6,6360


VERDICT #3: OPPOSITE - decline FALLS with age (62.7% -> 51.5% -> 42.6%) vs base 54.2%; the story 'older content decays more' runs against this slice.

Freshness buckets (sample-size floor 500 - under it, no verdict):


,decline_rate,pages,readable
fresh_bucket,,,
0-30,0.511,20480,True
31-90,0.589,175,False
91-180,0.611,9171,True
181-365,0.467,169,False
365+,0.600,5,False


3 of 5 freshness buckets are under the floor: 'insufficient data', which is itself a finding - freshness does not read on this slice.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's
assumption?* The flag: **`HIGH_VOLUME_POOR_POSITION`** — flag pages with ≥ 500
impressions/day **and** average position ≥ 11 (ML-07's rule, thresholds unchanged). Its
assumption: *high volume sitting off page one is visibility actually at stake.* Test #1
(volume) is the half behind FlyRank's quick-win flag; test #2 (position) is the half this
cell re-checks against the rule's own receipts.

In [3]:
# ---- The flag-linked test: does HIGH_VOLUME_POOR_POSITION hold? ----------------
bm = json.loads(Path("work/outputs/baseline_metrics_latest.json").read_text(encoding="utf-8"))
flag = ((df["gsc_impressions_avg"] >= 500) & (df["position_avg"] >= 11)).to_numpy()
y = df["is_declining_label"].to_numpy()

linked = pd.DataFrame([
    {"group": "flagged (rule)", "n": int(flag.sum()),
     "declining": round(float(y[flag].mean()), 4)},
    {"group": "not flagged",    "n": int((~flag).sum()),
     "declining": round(float(y[~flag].mean()), 4)},
    {"group": "frame base rate", "n": len(df), "declining": round(float(y.mean()), 4)},
])
display(linked)

assert int(flag.sum()) == bm["flagged_pages"] == 1_074, "flagged count drifted from ML-07 receipts"
assert bm["total_pages"] == len(df) == 140_599, "frame drifted from ML-07 receipts"
assert bm["impressions_verdict"] == "CONFIRMED" and bm["position_verdict"] == "MIXED", \
    "ML-07 verdicts changed - re-audit before using this notebook's verdicts"
rate_flagged = float(y[flag].mean())
expected = pb["queue"]["archetype_decline_rate"]["high_volume_deep_position"]
assert abs(rate_flagged - expected) <= 0.005, f"flagged decline {rate_flagged:.4f} vs receipts {expected}"

print(f"flagged pages decline at {rate_flagged:.1%} vs {float(y[~flag].mean()):.1%} unflagged "
      f"(base {float(y.mean()):.1%}) - real lift, receipts match ML-07.")
print("FLAG-LINKED TEST: volume half CONFIRMED (test #1); position half MIXED (test #2) "
      "-> keep volume as evidence, keep position >= 11 as POLICY (a written assumption), "
      "and re-audit it if the verdict ever flips.")

,group,n,declining
0,flagged (rule),1074,0.7803
1,not flagged,139525,0.4625
2,frame base rate,140599,0.4649


flagged pages decline at 78.0% vs 46.2% unflagged (base 46.5%) - real lift, receipts match ML-07.
FLAG-LINKED TEST: volume half CONFIRMED (test #1); position half MIXED (test #2) -> keep volume as evidence, keep position >= 11 as POLICY (a written assumption), and re-audit it if the verdict ever flips.


## 4. What this means in practice

The rule earns its **volume** threshold — decline rises with impressions across every bucket that clears
the 500-page floor (the 2000+ bucket, n=186, is printed but not read), and flagged pages decline
at 78% against a 46.5% base. The **position** threshold does not
earn its keep as evidence (non-monotonic), so it stays exactly what it is: a written
assumption, inherited unchanged, flagged MIXED for re-audit on new data. And age — the story
the whole refresh category is sold on — runs *backwards* on the starter slice, which is why
the queue ranks on observed visibility at stake rather than staleness. A negative verdict here
is a win: it saved reviewer hours from a rule the data does not support.

In [4]:
# ---- Verdict summary + receipts cross-check -----------------------------------
verdicts = pd.DataFrame([
    {"#": 1, "signal": "impressions volume -> decline", "verdict": verdict_1,
     "evidence": "45.1 -> 54.4 -> 56.3 -> 61.8% (n = 119,836 / 17,658 / 2,919; 2000+ n=186 under floor, not read)"},
    {"#": 2, "signal": "average position >= 11 -> decline", "verdict": verdict_2,
     "evidence": "47.2 -> 48.0 -> 45.9 -> 42.5%, non-monotonic"},
    {"#": 3, "signal": "content age -> decline (starter CSV)", "verdict": verdict_3,
     "evidence": "62.7 -> 51.5 -> 42.6% (n = 12,272 / 11,368 / 6,360)"},
    {"#": "flag", "signal": "HIGH_VOLUME_POOR_POSITION assumption", "verdict": "SPLIT",
     "evidence": "volume CONFIRMED (policy keeps it); position MIXED (policy, not evidence)"},
])
display(verdicts)

assert bm["impressions_verdict"] == "CONFIRMED" and bm["position_verdict"] == "MIXED"
assert verdict_1 == "CONFIRMED" and verdict_2 == "MIXED" and verdict_3 == "OPPOSITE"
assert imp_table.drop(index="2000+")["n"].min() >= 500 and imp_table.loc["2000+", "n"] < 500, \
    "impressions floor handling changed - re-read the buckets"
assert (pos_table["n"] >= 500).all() and (age_tbl["pages"] >= 500).all(), \
    "a verdict rests on a bucket under the floor"
print("SIGNAL AUDIT PASS - 3 verdicts recorded, floors respected, flag-linked receipts "
      "match baseline_metrics_latest.json and playbook_metrics.json.")

,#,signal,verdict,evidence
0,1,impressions volume -> decline,CONFIRMED,"45.1 -> 54.4 -> 56.3 -> 61.8% (n = 119,836 / 1..."
1,2,average position >= 11 -> decline,MIXED,"47.2 -> 48.0 -> 45.9 -> 42.5%, non-monotonic"
2,3,content age -> decline (starter CSV),OPPOSITE,"62.7 -> 51.5 -> 42.6% (n = 12,272 / 11,368 / 6..."
3,flag,HIGH_VOLUME_POOR_POSITION assumption,SPLIT,volume CONFIRMED (policy keeps it); position M...


SIGNAL AUDIT PASS - 3 verdicts recorded, floors respected, flag-linked receipts match baseline_metrics_latest.json and playbook_metrics.json.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.